# Resolución — Ejercicio SQL + pandas
### Unidad 4: SQL para Ciencia de Datos

Este notebook contiene la **resolución completa** del ejercicio `02_ejercicio_alumnos.ipynb`.
Está pensado como material del **docente** (o para que el alumno se autocorrija después de
intentarlo).

## 1. Preparar el entorno y los datos

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')

pedidos = pd.DataFrame({
    'id_pedido':  list(range(1, 13)),
    'producto':   ['Notebook', 'Mouse', 'Teclado', 'Monitor', 'Notebook', 'Auriculares',
                    'Mouse', 'Monitor', 'Webcam', 'Teclado', 'Monitor', 'Notebook'],
    'categoria':  ['Computación', 'Accesorios', 'Accesorios', 'Computación', 'Computación',
                    'Accesorios', 'Accesorios', 'Computación', 'Accesorios', 'Accesorios',
                    'Computación', 'Computación'],
    'precio':     [450000, 8500, 15000, 210000, 480000, 22000, 7900, 195000, 18000, 16500, 220000, 460000],
    'cantidad':   [1, 3, 2, 1, 1, 2, 4, 1, 1, 2, 1, 1],
    'ciudad':     ['Buenos Aires', 'Córdoba', 'Buenos Aires', 'Rosario', 'Córdoba',
                    'Buenos Aires', 'Rosario', 'Córdoba', 'Buenos Aires', 'Rosario',
                    'Córdoba', 'Buenos Aires']
})

pedidos.to_sql('pedidos', conn, index=False, if_exists='replace')
pedidos

,id_pedido,producto,categoria,precio,cantidad,ciudad
0,1,Notebook,Computación,450000,1,Buenos Aires
1,2,Mouse,Accesorios,8500,3,Córdoba
2,3,Teclado,Accesorios,15000,2,Buenos Aires
3,4,Monitor,Computación,210000,1,Rosario
4,5,Notebook,Computación,480000,1,Córdoba
5,6,Auriculares,Accesorios,22000,2,Buenos Aires
6,7,Mouse,Accesorios,7900,4,Rosario
7,8,Monitor,Computación,195000,1,Córdoba
8,9,Webcam,Accesorios,18000,1,Buenos Aires
9,10,Teclado,Accesorios,16500,2,Rosario


## 2. Ejercicio 1 — `SELECT` + `WHERE`

In [2]:
# --- Versión SQL ---
query_ej1 = '''
SELECT producto, precio, ciudad
FROM pedidos
WHERE categoria = 'Computación'
  AND precio > 200000
ORDER BY precio DESC;
'''

df_ej1_sql = pd.read_sql_query(query_ej1, conn)
df_ej1_sql

,producto,precio,ciudad
0,Notebook,480000,Córdoba
1,Notebook,460000,Buenos Aires
2,Notebook,450000,Buenos Aires
3,Monitor,220000,Córdoba
4,Monitor,210000,Rosario


In [3]:
# --- Versión pandas ---
df_ej1_pandas = (
    pedidos[(pedidos['categoria'] == 'Computación') & (pedidos['precio'] > 200000)]
    [['producto', 'precio', 'ciudad']]
    .sort_values('precio', ascending=False)
    .reset_index(drop=True)
)
df_ej1_pandas

,producto,precio,ciudad
0,Notebook,480000,Córdoba
1,Notebook,460000,Buenos Aires
2,Notebook,450000,Buenos Aires
3,Monitor,220000,Córdoba
4,Monitor,210000,Rosario


## 3. Ejercicio 2 — `GROUP BY` + agregaciones

In [4]:
# --- Versión SQL ---
query_ej2 = '''
SELECT
    categoria,
    SUM(precio * cantidad)      AS total_facturado,
    COUNT(*)                    AS cant_pedidos,
    ROUND(AVG(precio), 2)       AS precio_promedio
FROM pedidos
GROUP BY categoria
HAVING COUNT(*) > 4
ORDER BY total_facturado DESC;
'''

df_ej2_sql = pd.read_sql_query(query_ej2, conn)
df_ej2_sql

,categoria,total_facturado,cant_pedidos,precio_promedio
0,Computación,2015000,6,335833.33
1,Accesorios,182100,6,14650.00


In [5]:
# --- Versión pandas ---
df_ej2_pandas = (
    pedidos.assign(total=pedidos['precio'] * pedidos['cantidad'])
    .groupby('categoria')
    .agg(total_facturado=('total', 'sum'),
         cant_pedidos=('total', 'count'),
         precio_promedio=('precio', 'mean'))
    .round(2)
    .query('cant_pedidos > 4')
    .sort_values('total_facturado', ascending=False)
    .reset_index()
)
df_ej2_pandas

,categoria,total_facturado,cant_pedidos,precio_promedio
0,Computación,2015000,6,335833.33
1,Accesorios,182100,6,14650.00


## 4. Ejercicio 3 — Integrador

In [6]:
# --- Versión SQL ---
query_ej3 = '''
SELECT ciudad
FROM pedidos
WHERE categoria = 'Accesorios'
GROUP BY ciudad
ORDER BY SUM(cantidad) DESC
LIMIT 1;
'''

ciudad_top_sql = pd.read_sql_query(query_ej3, conn)['ciudad'].iloc[0]
ciudad_top_sql

'Rosario'

In [7]:
# --- Versión pandas ---
ciudad_top_pandas = (
    pedidos[pedidos['categoria'] == 'Accesorios']
    .groupby('ciudad')['cantidad']
    .sum()
    .idxmax()
)
ciudad_top_pandas

'Rosario'

## 5. Verificación

In [8]:
assert df_ej1_sql.reset_index(drop=True).equals(df_ej1_pandas)
assert df_ej2_sql.reset_index(drop=True).equals(df_ej2_pandas.reset_index(drop=True))
assert ciudad_top_sql == ciudad_top_pandas
print("✅ Ejercicios completos y consistentes")

✅ Ejercicios completos y consistentes


In [9]:
conn.close()